# Estimación FOPDT 
Este notebook realiza una versión mínima: carga datos, estima un modelo FOPDT con tres métodos (Ziegler–Nichols, Smith y optimización) y muestra gráficos y parámetros (K, τ, θ).


## Entrada de datos
El notebook carga automáticamente datos de `data_Q1_Q2.txt` e intenta detectar automáticamente columnas de tiempo, entrada y salida. Use el parámetro `canal` para seleccionar entre `'Q1'`, `'Q2'` o `'ambos'`. Cambie `data_file`, `u_col` o `y_col` si desea otra columna del fichero.

In [ ]:
# Imports básicos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

# Ruta por defecto (ajustable)
data_file = 'lab 1/rta/data_Q1.txt'

# Seleccionar canal: 'Q1', 'Q2' o 'ambos'
canal = 'Q1'  # Cambiar a 'Q2' o 'ambos' según desee

# Columnas por defecto: se elige automáticamente si encuentra nombres evidentes
u_col = None
y_col = None
t_col = None

# Cargar con pandas (comas como separador y encabezado en la primera línea)
# Usar encoding latin1 para manejar símbolos como ° o codificaciones locales
df = pd.read_csv(data_file, sep=',', header=0, comment='#', encoding='latin1')
cols = list(df.columns)
print('Columnas detectadas:', cols)

# Función para procesar un canal específico
def procesar_canal(df, canal):
    """Procesa los datos de un canal específico (Q1, Q2)."""
    input_candidates = []
    output_candidates = []
    t_col = None
    
    for c in cols:
        cl = c.lower()
        if 'tiempo' in cl or 'time' in cl:
            t_col = c
        # Filtrar por número de canal (1 para Q1, 2 para Q2)
        canal_num = '1' if canal == 'Q1' else '2'
        if ('cal' in cl or 'heater' in cl or 'input' in cl) and canal_num in cl:
            input_candidates.append(c)
        if ('temp' in cl or 'temper' in cl or 'y' == cl.strip()) and canal_num in cl:
            output_candidates.append(c)
    
    # Fallback: si no encuentra por número, usar candidatos generales
    if not input_candidates:
        for c in cols:
            cl = c.lower()
            if 'cal' in cl or 'set' in cl or 'heater' in cl or 'input' in cl:
                input_candidates.append(c)
    if not output_candidates:
        for c in cols:
            cl = c.lower()
            if 'temp' in cl or 'temper' in cl or 'y' == cl.strip():
                output_candidates.append(c)
    
    # Escoger la candidata con mayor variación si hay varias opciones
    if input_candidates:
        u_col = max(input_candidates, key=lambda c: np.nanmax(df[c].values.astype(float)) - np.nanmin(df[c].values.astype(float)))
    else:
        u_col = cols[1] if len(cols) > 1 else cols[0]
    
    if output_candidates:
        y_col = max(output_candidates, key=lambda c: np.nanmax(df[c].values.astype(float)) - np.nanmin(df[c].values.astype(float)))
    else:
        y_col = cols[-1]
    
    if t_col is None:
        t_col = cols[0]
    
    return t_col, u_col, y_col

# Procesar según el canal seleccionado
if canal == 'ambos':
    # Procesaré ambos canales en un diccionario
    print(f"\n=== Procesando AMBOS canales ===")
    datos_multiples = {}
    for ch in ['Q1', 'Q2']:
        t_col, u_col, y_col = procesar_canal(df, ch)
        t = df[t_col].values.astype(float)
        u = df[u_col].values.astype(float)
        y = df[y_col].values.astype(float)
        
        # Detección de escalón en la entrada (primer cambio significativo)
        du = np.diff(u)
        thres = (np.nanmax(u) - np.nanmin(u)) * 0.1
        step_idx_candidates = np.where(np.abs(du) > thres)[0]
        if len(step_idx_candidates) == 0:
            step_idx = 0
        else:
            step_idx = step_idx_candidates[0] + 1
        t_step = t[step_idx]
        
        # Valores inicial y final (promedios)
        pre_window = min(20, step_idx) if step_idx > 0 else min(20, len(y))
        post_window = min(20, len(y))
        y0 = np.mean(y[max(0, step_idx - pre_window):step_idx]) if step_idx > 0 else np.mean(y[:post_window])
        y_ss = np.mean(y[-post_window:])
        u0 = np.mean(u[max(0, step_idx - pre_window):step_idx]) if step_idx > 0 else np.mean(u[:post_window])
        u_ss = np.mean(u[-post_window:])
        delta_u = u_ss - u0 if (u_ss - u0) != 0 else 1.0
        
        datos_multiples[ch] = dict(t=t, u=u, y=y, t_step=t_step, y0=y0, y_ss=y_ss, u0=u0, u_ss=u_ss, delta_u=delta_u,
                                   t_col=t_col, u_col=u_col, y_col=y_col)
        print(f'{ch}: Tiempo=`{t_col}`, Entrada=`{u_col}`, Salida=`{y_col}`')
        print(f'{ch}: Escalón en índice {step_idx}, delta_u = {delta_u}')
    
    # Usar Q1 como principal para las celdas posteriores, y guardar ambos
    data = datos_multiples['Q1']
    data['ambos'] = datos_multiples
    
    # Gráfica con AMBOS canales
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Q1 - Salida e Entrada
    ax = axes[0, 0]
    ax.plot(datos_multiples['Q1']['t'], datos_multiples['Q1']['y'], label='Salida Q1')
    ax.plot(datos_multiples['Q1']['t'], datos_multiples['Q1']['u'], label='Entrada Q1')
    ax.axvline(datos_multiples['Q1']['t_step'], color='k', linestyle='--', alpha=0.5)
    ax.legend()
    ax.set_xlabel('Tiempo (s)')
    ax.set_title('Canal Q1')
    ax.grid(True, alpha=0.3)
    
    # Q2 - Salida e Entrada
    ax = axes[0, 1]
    ax.plot(datos_multiples['Q2']['t'], datos_multiples['Q2']['y'], label='Salida Q2', color='orange')
    ax.plot(datos_multiples['Q2']['t'], datos_multiples['Q2']['u'], label='Entrada Q2', color='red')
    ax.axvline(datos_multiples['Q2']['t_step'], color='k', linestyle='--', alpha=0.5)
    ax.legend()
    ax.set_xlabel('Tiempo (s)')
    ax.set_title('Canal Q2')
    ax.grid(True, alpha=0.3)
    
    # Comparación - Salidas
    ax = axes[1, 0]
    ax.plot(datos_multiples['Q1']['t'], datos_multiples['Q1']['y'], label='Salida Q1')
    ax.plot(datos_multiples['Q2']['t'], datos_multiples['Q2']['y'], label='Salida Q2', linestyle='--')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('Temperatura (°C)')
    ax.set_title('Comparación de Salidas')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Comparación - Entradas
    ax = axes[1, 1]
    ax.plot(datos_multiples['Q1']['t'], datos_multiples['Q1']['u'], label='Entrada Q1')
    ax.plot(datos_multiples['Q2']['t'], datos_multiples['Q2']['u'], label='Entrada Q2', linestyle='--')
    ax.set_xlabel('Tiempo (s)')
    ax.set_ylabel('Potencia (%)')
    ax.set_title('Comparación de Entradas')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

else:
    # Procesar un único canal
    print(f"\n=== Procesando canal {canal} ===")
    t_col, u_col, y_col = procesar_canal(df, canal)
    t = df[t_col].values.astype(float)
    u = df[u_col].values.astype(float)
    y = df[y_col].values.astype(float)
    
    # Detección de escalón en la entrada (primer cambio significativo)
    du = np.diff(u)
    thres = (np.nanmax(u) - np.nanmin(u)) * 0.1
    step_idx_candidates = np.where(np.abs(du) > thres)[0]
    if len(step_idx_candidates) == 0:
        step_idx = 0
    else:
        step_idx = step_idx_candidates[0] + 1
    t_step = t[step_idx]
    print(f'Usando columnas: tiempo=`{t_col}`, entrada=`{u_col}`, salida=`{y_col}`')
    print(f'Índice de escalón detectado: {step_idx}, tiempo escalón= {t_step}')
    
    # Valores inicial y final (promedios)
    pre_window = min(20, step_idx) if step_idx > 0 else min(20, len(y))
    post_window = min(20, len(y))
    y0 = np.mean(y[max(0, step_idx - pre_window):step_idx]) if step_idx > 0 else np.mean(y[:post_window])
    y_ss = np.mean(y[-post_window:])
    u0 = np.mean(u[max(0, step_idx - pre_window):step_idx]) if step_idx > 0 else np.mean(u[:post_window])
    u_ss = np.mean(u[-post_window:])
    delta_u = u_ss - u0 if (u_ss - u0) != 0 else 1.0
    print(f'delta_u = {delta_u}')
    
    # Guardar variables en el entorno para las siguientes celdas
    data = dict(t=t, u=u, y=y, t_step=t_step, y0=y0, y_ss=y_ss, u0=u0, u_ss=u_ss, delta_u=delta_u)
    
    # Gráfica para un único canal
    plt.figure(figsize=(8,3))
    plt.plot(data['t'], data['y'], label='Salida')
    plt.plot(data['t'], data['u'], label='Entrada (u)')
    plt.axvline(data['t_step'], color='k', linestyle='--', alpha=0.5)
    plt.legend()
    plt.xlabel('Tiempo (s)')
    plt.title(f'Canal {canal}')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

ModuleNotFoundError: No module named 'numpy'

## 6.1 Ziegler–Nichols (método de la tangente)
Procedimiento:
1) Trazar tangente en el punto de máxima pendiente de la respuesta.
2) Calcular intersección de esa tangente con la línea del valor inicial y con el valor final para obtener θ y una aproximación de τ.
3) Calcular K = Δy / Δu.

In [ ]:
# Implementación del método de la tangente (Ziegler–Nichols)
def metodo_zn(t, y, u, t_step, y0, y_ss, delta_u):
    """Calcula parámetros FOPDT usando método Ziegler–Nichols."""
    # derivada numérica
    dy_dt = np.gradient(y, t)
    # suavizar derivada (media móvil corta)
    window = max(1, int(len(dy_dt)*0.01))
    if window > 1:
        dy_dt_s = np.convolve(dy_dt, np.ones(window)/window, mode='same')
    else:
        dy_dt_s = dy_dt
    
    i_inf = np.nanargmax(np.abs(dy_dt_s))
    t_inf = t[i_inf]
    y_inf = y[i_inf]
    slope = dy_dt_s[i_inf]
    
    # Tangente: y = y_inf + slope*(t - t_inf)
    # Intersección con valor inicial y final
    t_intersect_init = t_inf + (y0 - y_inf)/slope
    t_intersect_final = t_inf + (y_ss - y_inf)/slope
    theta_zn = max(0.0, t_intersect_init - t_step)
    tau_zn = max(1e-6, t_intersect_final - t_intersect_init)
    K_zn = (y_ss - y0) / delta_u
    
    return dict(K=K_zn, tau=tau_zn, theta=theta_zn, t_inf=t_inf, y_inf=y_inf, slope=slope)

t = data['t']
y = data['y']
u = data['u']
t_step = data['t_step']
y0 = data['y0']
y_ss = data['y_ss']
delta_u = data['delta_u']

# Calcular para el canal principal
zn = metodo_zn(t, y, u, t_step, y0, y_ss, delta_u)
print(f'Ziegler–Nichols estimado: K={zn["K"]:.4f}, tau={zn["tau"]:.4f}, theta={zn["theta"]:.4f}')

# Si hay ambos canales, calcular también para el otro
if 'ambos' in data:
    print("\n=== Resultados Ziegler–Nichols para AMBOS canales ===")
    zn_ambos = {}
    for ch in ['Q1', 'Q2']:
        d = data['ambos'][ch]
        zn_ch = metodo_zn(d['t'], d['y'], d['u'], d['t_step'], d['y0'], d['y_ss'], d['delta_u'])
        zn_ambos[ch] = zn_ch
        print(f'{ch}: K={zn_ch["K"]:.4f}, tau={zn_ch["tau"]:.4f}, theta={zn_ch["theta"]:.4f}')
    data['zn_ambos'] = zn_ambos
    
    # Gráfica comparativa para ambos canales
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    for idx, ch in enumerate(['Q1', 'Q2']):
        d = data['ambos'][ch]
        zn_ch = zn_ambos[ch]
        ax = axes[idx]
        
        ax.plot(d['t'], d['y'], label='Salida medida', linewidth=2)
        # tangente
        y_tan = zn_ch['y_inf'] + zn_ch['slope']*(d['t'] - zn_ch['t_inf'])
        ax.plot(d['t'], y_tan, '--', label='Tangente (inflection)', linewidth=1.5)
        ax.axvline(d['t_step'] + zn_ch['theta'], color='C2', linestyle=':', label=f'theta={zn_ch["theta"]:.3f}', linewidth=2)
        ax.scatter([zn_ch['t_inf']], [zn_ch['y_inf']], c=['C3'], s=80, zorder=5)
        ax.legend(fontsize=9)
        ax.set_xlabel('Tiempo (s)', fontsize=10)
        ax.set_ylabel('Temperatura (°C)', fontsize=10)
        ax.set_title(f'Canal {ch} - Ziegler–Nichols\nK={zn_ch["K"]:.4f}, τ={zn_ch["tau"]:.4f}', fontsize=11)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    # Gráfica para un único canal
    plt.figure(figsize=(8, 4))
    plt.plot(t, y, label='Salida medida', linewidth=2)
    # tangente
    y_tan = zn['y_inf'] + zn['slope']*(t - zn['t_inf'])
    plt.plot(t, y_tan, '--', label='Tangente (inflection)', linewidth=1.5)
    plt.axvline(t_step + zn['theta'], color='C2', linestyle=':', label='theta (ZN)', linewidth=2)
    plt.scatter([zn['t_inf']], [zn['y_inf']], c=['C3'], s=80, zorder=5)
    plt.legend(fontsize=10)
    plt.xlabel('Tiempo (s)', fontsize=11)
    plt.ylabel('Temperatura (°C)', fontsize=11)
    plt.title('Ziegler–Nichols: tangente en inflexión', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Guardar estimado principal
data['zn'] = zn


## 6.2 Smith
Procedimiento:
- Se traza la misma tangente y se obtiene θ como intersección.
- τ se estima como el tiempo que tarda en alcanzar el 63.2% del cambio final después de la demora (método 0.632).

**Explicación (por qué 63.2%)**:
Para un sistema FOPDT la respuesta al escalón es $y(t)=y_0+K\Delta u(1-e^{-(t-\theta)/\tau})$ para $t>\theta$. Evaluando en $t=\theta+\tau$ se obtiene $1-e^{-1}\approx0.632$. Por eso se usa el 63.2%: el tiempo en que la salida alcanza ese porcentaje (restando la demora) ofrece una estimación directa de $\tau$.

Nota: en datos ruidosos o con muestreo escaso conviene suavizar la señal o elegir la intersección más representativa; en este notebook usamos $\theta$ de la tangente y luego el cruce al 63.2% para calcular $\tau$.

In [ ]:
# Método Smith (tangente + 63.2%)
def metodo_smith(t, y, u, t_step, y0, y_ss, delta_u, zn_dict):
    """Calcula parámetros FOPDT usando método Smith."""
    theta_smith = zn_dict['theta']
    y_target = y0 + 0.632*(y_ss - y0)
    # buscar tiempo en que la señal cruza el 63.2% (primera vez)
    idx_632 = np.where(y >= y_target)[0]
    if len(idx_632) == 0:
        t_632 = zn_dict['t_inf'] + zn_dict['tau']  # fallback
    else:
        t_632 = t[idx_632[0]]
    tau_smith = max(1e-6, t_632 - (t_step + theta_smith))
    K_smith = (y_ss - y0) / delta_u
    
    return dict(K=K_smith, tau=tau_smith, theta=theta_smith, y_target=y_target)

# Calcular para el canal principal
smith = metodo_smith(t, y, u, t_step, y0, y_ss, delta_u, data['zn'])
print(f'Smith estimado: K={smith["K"]:.4f}, tau={smith["tau"]:.4f}, theta={smith["theta"]:.4f}')

# Si hay ambos canales, calcular también para el otro
if 'ambos' in data:
    print("\n=== Resultados Smith para AMBOS canales ===")
    smith_ambos = {}
    for ch in ['Q1', 'Q2']:
        d = data['ambos'][ch]
        smith_ch = metodo_smith(d['t'], d['y'], d['u'], d['t_step'], d['y0'], d['y_ss'], d['delta_u'], data['zn_ambos'][ch])
        smith_ambos[ch] = smith_ch
        print(f'{ch}: K={smith_ch["K"]:.4f}, tau={smith_ch["tau"]:.4f}, theta={smith_ch["theta"]:.4f}')
    data['smith_ambos'] = smith_ambos
    
    # Gráfica comparativa para ambos canales
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    for idx, ch in enumerate(['Q1', 'Q2']):
        d = data['ambos'][ch]
        smith_ch = smith_ambos[ch]
        ax = axes[idx]
        
        ax.plot(d['t'], d['y'], label='Salida medida', linewidth=2)
        ax.axvline(d['t_step'] + smith_ch['theta'], color='C2', linestyle=':', label=f'theta={smith_ch["theta"]:.3f}', linewidth=2)
        ax.axhline(smith_ch['y_target'], color='C3', linestyle='--', label=f'63.2% (y_target={smith_ch["y_target"]:.2f})', linewidth=1.5)
        ax.legend(fontsize=9)
        ax.set_xlabel('Tiempo (s)', fontsize=10)
        ax.set_ylabel('Temperatura (°C)', fontsize=10)
        ax.set_title(f'Canal {ch} - Smith\nK={smith_ch["K"]:.4f}, τ={smith_ch["tau"]:.4f}', fontsize=11)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    # Gráfica para un único canal
    plt.figure(figsize=(8, 4))
    plt.plot(t, y, label='Salida medida', linewidth=2)
    plt.axvline(t_step + smith['theta'], color='C2', linestyle=':', label='theta (Smith)', linewidth=2)
    plt.axhline(smith['y_target'], color='C3', linestyle='--', label='63.2%', linewidth=1.5)
    plt.legend(fontsize=10)
    plt.xlabel('Tiempo (s)', fontsize=11)
    plt.ylabel('Temperatura (°C)', fontsize=11)
    plt.title('Smith: 0.632 method', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Guardar estimado principal
data['smith'] = smith


## 6.3 Comparación y Optimización (identificación por mínimos cuadrados)
Usamos los estimados anteriores como iniciales para una optimización no lineal que minimiza el error entre la respuesta del FOPDT y los datos experimentales.

In [ ]:
# Modelo FOPDT vectorizado para respuesta a escalón
def fopdt_response(t, K, tau, theta, t_step, u0, delta_u, y0):
    t_rel = t - t_step - theta
    y_mod = np.where(t_rel > 0, y0 + K * delta_u * (1.0 - np.exp(-t_rel / tau)), y0)
    return y_mod

# Residuals para least_squares
def residuals(p, t, y, t_step, u0, delta_u, y0):
    K, tau, theta = p
    if tau <= 0 or theta < 0 or not np.all(np.isfinite(p)):
        return 1e6 * np.ones_like(y)
    y_mod = fopdt_response(t, K, tau, theta, t_step, u0, delta_u, y0)
    return y_mod - y

def optimizar_fopdt(t, y, u, t_step, u0, delta_u, y0, smith_dict):
    """Optimiza parámetros FOPDT usando least_squares."""
    # Condiciones iniciales a partir de Smith
    p0 = np.array([smith_dict['K'], smith_dict['tau'], smith_dict['theta']], dtype=float)
    # límites razonables: K libre, tau>1e-6, theta>=0
    lb = np.array([-np.inf, 1e-6, 0.0], dtype=float)
    ub = np.array([np.inf, t[-1] * 2, t[-1]], dtype=float)
    
    # Si no es finita, usar fallback
    if not np.all(np.isfinite(p0)):
        p0 = np.array([1.0, max(1.0, 0.1 * (t[-1] - t[0])), 0.0], dtype=float)
    
    # Asegurar que la semilla inicial esté dentro de los límites
    p0 = np.clip(p0, lb, ub)
    p0[1] = max(p0[1], lb[1] + 1e-6)
    p0[2] = max(p0[2], lb[2])
    
    res = least_squares(residuals, p0, bounds=(lb, ub), args=(t, y, t_step, u0, delta_u, y0))
    K_opt, tau_opt, theta_opt = res.x
    
    return dict(K=K_opt, tau=tau_opt, theta=theta_opt)

# Optimizar para el canal principal
opt = optimizar_fopdt(t, y, u, t_step, data['u0'], data['delta_u'], data['y0'], data['smith'])
print(f'Optimización (least_squares): K={opt["K"]:.4f}, tau={opt["tau"]:.4f}, theta={opt["theta"]:.4f}')

# Si hay ambos canales, optimizar también para el otro
if 'ambos' in data:
    print("\n=== Resultados Optimización para AMBOS canales ===")
    opt_ambos = {}
    for ch in ['Q1', 'Q2']:
        d = data['ambos'][ch]
        opt_ch = optimizar_fopdt(d['t'], d['y'], d['u'], d['t_step'], d['u0'], d['delta_u'], d['y0'], data['smith_ambos'][ch])
        opt_ambos[ch] = opt_ch
        print(f'{ch}: K={opt_ch["K"]:.4f}, tau={opt_ch["tau"]:.4f}, theta={opt_ch["theta"]:.4f}')
    data['opt_ambos'] = opt_ambos
    
    # Gráfica comparativa para ambos canales
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    for idx, ch in enumerate(['Q1', 'Q2']):
        d = data['ambos'][ch]
        opt_ch = opt_ambos[ch]
        y_fit = fopdt_response(d['t'], opt_ch['K'], opt_ch['tau'], opt_ch['theta'], d['t_step'], d['u0'], d['delta_u'], d['y0'])
        ax = axes[idx]
        
        ax.plot(d['t'], d['y'], label='Salida medida', linewidth=2)
        ax.plot(d['t'], y_fit, '--', label='FOPDT optimizado', linewidth=2, alpha=0.8)
        ax.legend(fontsize=10)
        ax.set_xlabel('Tiempo (s)', fontsize=10)
        ax.set_ylabel('Temperatura (°C)', fontsize=10)
        ax.set_title(f'Canal {ch} - Optimización\nK={opt_ch["K"]:.4f}, τ={opt_ch["tau"]:.4f}, θ={opt_ch["theta"]:.4f}', fontsize=11)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    # Gráfica para un único canal
    y_fit = fopdt_response(t, opt['K'], opt['tau'], opt['theta'], t_step, data['u0'], data['delta_u'], data['y0'])
    plt.figure(figsize=(8, 4))
    plt.plot(t, y, label='Salida medida', linewidth=2)
    plt.plot(t, y_fit, '--', label='FOPDT optimizado', linewidth=2, alpha=0.8)
    plt.legend(fontsize=10)
    plt.xlabel('Tiempo (s)', fontsize=11)
    plt.ylabel('Temperatura (°C)', fontsize=11)
    plt.title('Comparación: datos vs FOPDT optimizado', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Guardar estimado principal
data['opt'] = opt


## Resumen de estimaciones
Se imprimen las tres estimaciones para fácil comparación.

In [ ]:
print('Ziegler–Nichols: ', data['zn'])
print('Smith: ', data['smith'])
print('Optimización: ', data['opt'])

In [ ]:
print("="*60)
print("RESUMEN DE ESTIMACIONES FOPDT")
print("="*60)

# Mostrar resultados del canal principal (Q1 o el seleccionado)
print("\n### CANAL PRINCIPAL ###\n")
print("=== Ziegler–Nichols ===")
print(f"K     : {data['zn']['K']:.6f}")
print(f"tau   : {data['zn']['tau']:.6f}")
print(f"theta : {data['zn']['theta']:.6f}")

print("\n=== Smith ===")
print(f"K     : {data['smith']['K']:.6f}")
print(f"tau   : {data['smith']['tau']:.6f}")
print(f"theta : {data['smith']['theta']:.6f}")

print("\n=== Optimización ===")
print(f"K     : {data['opt']['K']:.6f}")
print(f"tau   : {data['opt']['tau']:.6f}")
print(f"theta : {data['opt']['theta']:.6f}")

# Si se procesaron ambos canales, mostrar también Q2
if 'ambos' in data and 'zn_ambos' in data:
    print("\n" + "="*60)
    print("### CANAL Q2 (cuando se elige 'ambos') ###\n")
    
    print("=== Ziegler–Nichols ===")
    print(f"K     : {data['zn_ambos']['Q2']['K']:.6f}")
    print(f"tau   : {data['zn_ambos']['Q2']['tau']:.6f}")
    print(f"theta : {data['zn_ambos']['Q2']['theta']:.6f}")
    
    print("\n=== Smith ===")
    print(f"K     : {data['smith_ambos']['Q2']['K']:.6f}")
    print(f"tau   : {data['smith_ambos']['Q2']['tau']:.6f}")
    print(f"theta : {data['smith_ambos']['Q2']['theta']:.6f}")
    
    print("\n=== Optimización ===")
    print(f"K     : {data['opt_ambos']['Q2']['K']:.6f}")
    print(f"tau   : {data['opt_ambos']['Q2']['tau']:.6f}")
    print(f"theta : {data['opt_ambos']['Q2']['theta']:.6f}")
    
    # Tabla comparativa
    print("\n" + "="*60)
    print("### TABLA COMPARATIVA (Q1 vs Q2) ###\n")
    print(f"{'Método':<20} {'Parámetro':<12} {'Q1':<15} {'Q2':<15}")
    print("-"*62)
    
    for method, key in [('Ziegler–Nichols', 'zn'), ('Smith', 'smith'), ('Optimización', 'opt')]:
        for param in ['K', 'tau', 'theta']:
            q1_val = data[key][param]
            q2_val = data[f'{key}_ambos']['Q2'][param]
            print(f"{method:<20} {param:<12} {q1_val:<15.6f} {q2_val:<15.6f}")
        print("-"*62)

print("\n" + "="*60)

In [ ]:
!pip install control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# MODELOS FOPDT
# G(s) = K/(tau*s + 1) * e^(-theta*s)
# ==========================================================

# -------- Ziegler-Nichols --------
K_zn = data['zn']['K']
tau_zn = data['zn']['tau']
theta_zn = data['zn']['theta']

# -------- Smith --------
K_smith = data['smith']['K']
tau_smith = data['smith']['tau']
theta_smith = data['smith']['theta']

# -------- Optimización --------
K_opt = data['opt']['K']
tau_opt = data['opt']['tau']
theta_opt = data['opt']['theta']

# ==========================================================
# RESPUESTA DEL MODELO
# ==========================================================

def respuesta_fopdt(t, K, tau, theta, t_step, u0, delta_u, y0):
    t_rel = t - t_step - theta
    return np.where(t_rel > 0, y0 + K * delta_u * (1.0 - np.exp(-t_rel / tau)), y0)

# ==========================================================
# GRÁFICA
# ==========================================================

if 'ambos' in data and 'zn_ambos' in data and 'smith_ambos' in data and 'opt_ambos' in data:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

    ax = axes[0]
    ax.plot(data['t'], data['y'], color='black', linewidth=2.5, alpha=0.8, label='Dato original')
    ax.plot(data['t'], respuesta_fopdt(data['t'], K_zn, tau_zn, theta_zn, data['t_step'], data['u0'], data['delta_u'], data['y0']), label='Ziegler-Nichols')
    ax.plot(data['t'], respuesta_fopdt(data['t'], K_smith, tau_smith, theta_smith, data['t_step'], data['u0'], data['delta_u'], data['y0']), label='Smith')
    ax.plot(data['t'], respuesta_fopdt(data['t'], K_opt, tau_opt, theta_opt, data['t_step'], data['u0'], data['delta_u'], data['y0']), label='Optimización')
    ax.set_title('Respuesta al Escalón - Q1')
    ax.set_xlabel('Tiempo')
    ax.set_ylabel('Salida')
    ax.grid(True)
    ax.legend()

    d2 = data['ambos']['Q2']
    zn2 = data['zn_ambos']['Q2']
    smith2 = data['smith_ambos']['Q2']
    opt2 = data['opt_ambos']['Q2']

    ax = axes[1]
    ax.plot(d2['t'], d2['y'], color='black', linewidth=2.5, alpha=0.8, label='Dato original')
    ax.plot(d2['t'], respuesta_fopdt(d2['t'], zn2['K'], zn2['tau'], zn2['theta'], d2['t_step'], d2['u0'], d2['delta_u'], d2['y0']), label='Ziegler-Nichols')
    ax.plot(d2['t'], respuesta_fopdt(d2['t'], smith2['K'], smith2['tau'], smith2['theta'], d2['t_step'], d2['u0'], d2['delta_u'], d2['y0']), label='Smith')
    ax.plot(d2['t'], respuesta_fopdt(d2['t'], opt2['K'], opt2['tau'], opt2['theta'], d2['t_step'], d2['u0'], d2['delta_u'], d2['y0']), label='Optimización')
    ax.set_title('Respuesta al Escalón - Q2')
    ax.set_xlabel('Tiempo')
    ax.grid(True)
    ax.legend()

    plt.tight_layout()
    plt.show()
else:
    plt.figure(figsize=(10, 6))
    plt.plot(data['t'], data['y'], color='black', linewidth=2.5, alpha=0.8, label='Dato original')
    plt.plot(data['t'], respuesta_fopdt(data['t'], K_zn, tau_zn, theta_zn, data['t_step'], data['u0'], data['delta_u'], data['y0']), label='Ziegler-Nichols')
    plt.plot(data['t'], respuesta_fopdt(data['t'], K_smith, tau_smith, theta_smith, data['t_step'], data['u0'], data['delta_u'], data['y0']), label='Smith')
    plt.plot(data['t'], respuesta_fopdt(data['t'], K_opt, tau_opt, theta_opt, data['t_step'], data['u0'], data['delta_u'], data['y0']), label='Optimización')
    plt.title('Respuesta al Escalón - Modelos FOPDT')
    plt.xlabel('Tiempo')
    plt.ylabel('Salida')
    plt.grid(True)
    plt.legend()
    plt.show()

## Interpretación del modelo

Si se toma `Q1(t)` como la entrada y `T1(t)` como la salida, entonces el modelo asociado al canal queda escrito en el dominio de Laplace como:

$$G_{11}(s)=\frac{T_1(s)}{Q_1(s)} \approx \frac{K e^{-\theta s}}{\tau s + 1}$$

Aquí `Q1(s)` y `T1(s)` son las transformadas de Laplace de las señales en el tiempo. No se trata del dominio de frecuencia en sentido estricto; la interpretación de frecuencia aparece después, si se evalúa el modelo en $s=j\omega$.

In [ ]:
import control as ctrl
import matplotlib.pyplot as plt

#  - data['zn']   (ZN estimado para Q1)
#  - data['opt']  (Optimización para Q1)
#  - data['smith'](Smith para Q1)
modelo = data['zn']
K = modelo['K']
tau = modelo['tau']
theta = modelo['theta']

G11_sin_retraso = ctrl.TransferFunction([K], [tau, 1])
num_delay, den_delay = ctrl.pade(theta, 1)
G11 = G11_sin_retraso * ctrl.TransferFunction(num_delay, den_delay)

print('G11(s) =')
print(G11)

if 'respuesta_fopdt' in globals():
    t = data['t']
    y_model = respuesta_fopdt(
        t,
        K,
        tau,
        theta,
        data['t_step'],
        data['u0'],
        data['delta_u'],
        data['y0'],
    )

    plt.figure(figsize=(10, 4))
    plt.plot(t, data['y'], 'k', label='Datos experimentales')
    plt.plot(t, y_model, 'r--', label='G11 estimado')
    plt.grid(True)
    plt.legend()
    plt.xlabel('Tiempo [s]')
    plt.ylabel('Salida')
    plt.title('Comparación del canal G11')
    plt.show()


## Interpretación del modelo para G21

Si tomamos `Q1(t)` como la entrada y `T2(t)` como la salida, el modelo del canal se puede escribir en Laplace como:

$$G_{21}(s)=\frac{T_2(s)}{Q_1(s)} \approx \frac{K e^{-\theta s}}{\tau s + 1}$$

Aquí `Q1(s)` y `T2(s)` son las transformadas de Laplace de las señales en el tiempo. La notación expresa de nuevo una FOPDT (ganancia estática `K`, constante de tiempo `\tau` y retardo `\theta`).

In [ ]:
import control as ctrl
import matplotlib.pyplot as plt

# === Seleccione el modelo para G21 cambiando solo esta línea ===
# Opciones típicas:
#  - data['zn_ambos']['Q2']   (ZN estimado para Q2)
#  - data['opt_ambos']['Q2']  (Optimización para Q2)
#  - data['smith_ambos']['Q2'](Smith para Q2)
#  - data['ambos']['Q2']      (usar parámetros incrustados en 'ambos', si existen)
#  - data['zn']              (fallback: usar el estimado principal)
modelo = data['zn_ambos']['Q2'] if 'zn_ambos' in data and 'Q2' in data['zn_ambos'] else data.get('zn', {})

# Datos experimentales del canal 2 (si existen)
d2 = data['ambos']['Q2'] if 'ambos' in data and 'Q2' in data['ambos'] else None

# Extraer parámetros
K = modelo['K']
tau = modelo['tau']
theta = modelo['theta']

print(f"Modelo seleccionado para G21: K={K:.6f}, tau={tau:.6f}, theta={theta:.6f}")

# Construir la TF con Padé para el retraso
G21_sin_retraso = ctrl.TransferFunction([K], [tau, 1])
num_delay, den_delay = ctrl.pade(theta, 1)
G21 = G21_sin_retraso * ctrl.TransferFunction(num_delay, den_delay)

print('G21(s) =')
print(G21)

# Comparación con datos experimentales si están disponibles
if d2 is not None and 't' in d2 and 'y' in d2:
    t = d2['t']
    if 'respuesta_fopdt' in globals():
        y_model = respuesta_fopdt(t, K, tau, theta,
                                  d2.get('t_step', data.get('t_step', 0)),
                                  d2.get('u0', data.get('u0', 0)),
                                  d2.get('delta_u', data.get('delta_u', 1)),
                                  d2.get('y0', data.get('y0', 0)))

        plt.figure(figsize=(10, 4))
        plt.plot(t, d2['y'], 'k', label='T2 datos experimentales')
        plt.plot(t, y_model, 'r--', label='G21 estimado')
        plt.grid(True)
        plt.legend()
        plt.xlabel('Tiempo [s]')
        plt.ylabel('Salida T2')
        plt.title('Estimación de G21 (T2/Q1)')
        plt.show()
